In [1]:
import json
import os, sys
import pickle
import datetime
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
from sklearn import metrics
import seaborn as sns
import ROOT as rt


## user define
sys.path.append(".")
sys.path.append("./Utils")
from Plotter import HistStack
from CommonTrain import *
from Model import Seq

2023-03-31 20:54:39.213007: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /cvmfs/sft.cern.ch/lcg/releases/MCGenerators/thepeg/2.2.3-6cb32/x86_64-centos7-gcc11-opt/lib/ThePEG:/cvmfs/sft.cern.ch/lcg/releases/MCGenerators/herwig++/7.2.3-90caf/x86_64-centos7-gcc11-opt/lib/Herwig:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/jaxlib/mlir/_mlir_libs:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/torch/lib:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/onnxruntime/capi/:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/tensorflow:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/tensorfl

Welcome to JupyROOT 6.26/08


# Set config

In [2]:
setting={}
with open('config/Settrain.json') as json_file:
    data = json.load(json_file)
    setting=data
INPUTDIR = setting["INPUTDIR"]
VERSION = setting["VERSION"]
# SIGNAL_MASSES = ["Zp-1700_CH-345", "Zp-2900_CH-345", "Zp-4100_CH-345", "Zp-2900_CH-1095", "Zp-4100_CH-1095", "Zp-4100_CH-1845"]
CHANNEL="ee"
# ERA="ee17"
print(VERSION)
INPUTDIR = sys.argv[1]
REGION = sys.argv[2]

# # for dev
# INPUTDIR = setting["INPUTDIR"]+'/ee18/'
# REGION = 'SR'

2803_equalpropv1dev


# Load data

In [3]:
if REGION == 'SR':
    file = 'nested_dic_df.pkl'
if REGION == 'VR':   
    file = 'nested_dic_df_PT.pkl'
# file = "/"+ERA+"/nested_dic_df.pkl"
with open(INPUTDIR+'/'+file, 'rb') as f:
    nested_dic_df = pickle.load(f)
 
dic_df=nested_dic_df['raw']
dic_df_met_u_unsort=nested_dic_df['met_u']
dic_df_met_d_unsort=nested_dic_df['met_d'] 

selected_columns = [
#                     'MC1C1_pre', 'H3PP_pre', 'H3TPP_pre', 
                    'pt_lead_pre', 'mass_pre', # by auc rank
                    'MT_pre', 'pt_trail_pre', 'Zpt_pre', 'dRll_pre', 'PuppiMET_pre',
                    'dPhilMET_pre', 'dPhillMET_pre', 'U_pre', 'dPhill_pre', 'MT2_pre',
                    'JZB_pre'] # order used in training

# # dic_df_meta={}
# # check b jet
# query_str="nbjet_pre==0"

# print("####################################################")
# print("####   (unweighted) #events after preselection  ####")
# print("####################################################")
# for key in dic_df:  
# #     dic_df_meta[key]=dic_df[key].query(query_str)
#     dic_df[key]=dic_df[key].query(query_str) #temp!
#     print(key, ":", dic_df[key].shape[0])
    
dic_df_train={}    
dic_df_met_u={}
dic_df_met_d={}
for key in dic_df:
    print(key)
    #dic_df_train[key]=pd.concat([dic_df[key][selected_columns], dic_df1[key][selected_columns]]) #when use ll channel
    #dic_df[key]=pd.concat([dic_df[key], dic_df1[key]])
#     dic_df_train[key]=dic_df_meta[key][selected_columns] # check b jet   
    dic_df_train[key]=dic_df[key][selected_columns]    
    dic_df_met_u[key]=dic_df_met_u_unsort[key][selected_columns]    
    dic_df_met_d[key]=dic_df_met_d_unsort[key][selected_columns]

Zp-4100_CH-1345
DY1500To2000
Zp-1700_CH-345
Zp-2500_CH-345
DY500To700
Zp-1700_CH-845
Zp-4100_CH-595
Zp-3300_CH-345
Zp-1700_CH-595
Zp-3700_CH-345
Zp-2100_CH-595
Zp-2900_CH-345
Zp-4100_CH-1095
Zp-2900_CH-845
Zp-3700_CH-1345
Zp-3300_CH-1595
Zp-2500_CH-595
Zp-4100_CH-345
Zp-3700_CH-1845
Zp-2500_CH-845
Zp-4100_CH-1595
Zp-4100_CH-1845
Zp-3300_CH-595
Zp-2100_CH-345
Zp-2900_CH-595
Zp-4100_CH-845
Zp-2100_CH-845
Zp-3300_CH-1345
Zp-3700_CH-1595
DY2000To3000
Zp-3700_CH-595
DY700To800
Zp-2900_CH-1345
Zp-3700_CH-1095
DY800To1000
DY1000To1500
Zp-3700_CH-845
Zp-3300_CH-1095
Zp-2900_CH-1095
TT
TTSemi
TTZ
DY
top
antitop
ZZ
WZ
WW
Data


In [4]:
print(dic_df["Zp-2500_CH-345"].columns)
print(dic_df_train["Zp-2500_CH-345"].columns)
print(dic_df_met_u["Zp-2500_CH-345"].columns)

Index(['PuppiMET_pre', 'mass_pre', 'pt_lead_pre', 'pt_trail_pre', 'U_pre',
       'JZB_pre', 'dPhill_pre', 'dPhilMET_pre', 'MT_pre', 'MT2_pre', 'Zpt_pre',
       'dRll_pre', 'dPhillMET_pre', 'MET_pre', 'MET_phi_pre',
       'PuppiMET_phi_pre', 'eta_lead_pre', 'eta_trail_pre', 'phi_lead_pre',
       'phi_trail_pre', 'njet_pre', 'nbjetflav_pre', 'genweight_u_pre',
       'genweight_d_pre', 'genweight_pre', 'puweight_pre', 'puweight_u_pre',
       'puweight_d_pre', 'l1prefiring_pre', 'l1prefiring_u_pre',
       'l1prefiring_d_pre', 'effSF_pre', 'effSF_u_pre', 'effSF_d_pre',
       'topSF_pre', 'topSF_u_pre', 'topSF_d_pre', 'btagSF_pre', 'weight',
       'wopu', 'wol1', 'woeff', 'wotop', 'wobsf', 'weightpuweightUp',
       'weightpuweightDown', 'weightl1prefiringUp', 'weightl1prefiringDown',
       'weighteffSFUp', 'weighteffSFDown', 'weighttopSFUp', 'weighttopSFDown',
       'weightgenweightUp', 'weightgenweightDown'],
      dtype='object')
Index(['pt_lead_pre', 'mass_pre', 'MT_pre', 'pt_

In [5]:
def get_x(sig_key, dic_df_train):
    # x: sig and all bkgs concatenated
    x = dic_df_train[sig_key].copy(deep=True)
    for k in dic_df_train:
        # filter data out
        if "Data" in k:
            continue
        if "Zp" not in k:
            x = np.concatenate([x, dic_df_train[k]], axis=0)
    
    return x

In [6]:
def get_xy(sig_key, dic_predict_sort):
    # x: sig and all bkgs concatenated
    x = dic_predict_sort[sig_key].copy(deep=True)
    for k in dic_predict_sort:
        # filter data out
        if "Data" in k:
            continue
        if "Zp" not in k:
            x = np.concatenate([x, dic_predict_sort[k]], axis=0)
            
    tot_sig = dic_predict_sort[sig_key].shape[0]
    tot_bkgs = x.shape[0] - tot_sig        
    y_sig = np.full((tot_sig,1), 1)
    y_bkgs = np.full((tot_bkgs,1), 0) 
    y = np.concatenate((y_sig, y_bkgs), axis=0) 
    
    return x, y

In [7]:
import math
sys_list = ["puweight", "l1prefiring", "effSF", "topSF"]
# sys_list = ["l1prefiring"]
if REGION == 'SR':
    threshold = 0.3
else:
    threshold = 1    
    
def draw_with_systematics(dic_df_raw, sys_list, variable, r, region):
    dic_sys = {}
    dic_sys_sqr = {}
    # nominal weight plot
    file_prefix = "Outputs/"+REGION
    hist=HistStack(dic_df_raw, variable, 'weight', r[0], r[1], r[2], r[3])            
    c1=hist.draw_hist("Zp-2500_CH-345", ERA, file_name=file_prefix+"/norm_dnnscore.pdf", blind_from=threshold)     
    dic_yields = hist.get_yields_by_bin()
    bkg_norm = dic_yields['bkg']

    for s in sys_list:
        # up
        hist=HistStack(dic_df_raw, variable, 'weight'+s+'Up', r[0], r[1], r[2], r[3])            
        c1=hist.draw_hist("Zp-2500_CH-345", ERA, file_name=file_prefix+"/sys/systUp_"+s+".pdf", blind_from=threshold)     
        dic_yields = hist.get_yields_by_bin()
        sysUp = dic_yields['bkg']
        dic_sys[s+'Up'] = np.array(sysUp) - np.array(bkg_norm)
        # down
        hist=HistStack(dic_df_raw, variable, 'weight'+s+'Down', r[0], r[1], r[2], r[3])            
        c1=hist.draw_hist("Zp-2500_CH-345", ERA, file_name=file_prefix+"/sys/systDown_"+s+".pdf", blind_from=threshold)     
        dic_yields = hist.get_yields_by_bin()
        sysDown = dic_yields['bkg']
        dic_sys[s+'Down'] = np.array(sysDown) - np.array(bkg_norm)
        
    # si^2
    for s in sys_list:
        # by source
        sqrUp = [x*x if x>0 else -x*x for x in dic_sys[s+'Up']]
        sqrUp_up = [x if x>0 else 0 for x in sqrUp]
        sqrUp_down = [x if x<0 else 0 for x in sqrUp]
        sqrDown = [x*x if x>0 else -x*x for x in dic_sys[s+'Down']]
        sqrDown_up = [x if x>0 else 0 for x in sqrDown]
        sqrDown_down = [x if x<0 else 0 for x in sqrDown]
        
        sqr_up = np.array(sqrUp_up) + np.array(sqrDown_up)
        sqr_down = np.array(sqrUp_down) + np.array(sqrDown_down)
        
        dic_sys_sqr[s+'Up'] = sqr_up
        dic_sys_sqr[s+'Down'] = np.fabs(sqr_down) #store absolute
        
    print(dic_sys)
    print(dic_sys_sqr)
    # sqrt(sumi si^2)
    for i, s in enumerate(sys_list):
        if i==0:
            sum_up = dic_sys_sqr[s+'Up'].copy() # be carefull not to use shallow copy
            sum_down = dic_sys_sqr[s+'Down'].copy()
        else:
            sum_up += dic_sys_sqr[s+'Up']
            sum_down += dic_sys_sqr[s+'Down']

    dic_sys_sqr['sum_up'] = np.sqrt(sum_up)
#     print(sum_up)
#     print(sum_down)
    dic_sys_sqr['sum_down'] = np.sqrt(sum_down)
    print(dic_sys_sqr)
            
    return dic_sys_sqr

In [8]:
from multiprocessing import Pool
def wrapper_mp(model_mass):
    
#     # dump roc
#     dic_roc = {}
#     x, y = get_xy(model_mass, dic_predict_sort)
#     tpr, fpr, auc = dump_roc(model_mass, y, x, dic_predict_sort)
#     dic_roc['tpr'] = tpr
#     dic_roc['fpr'] = fpr
#     dic_roc['auc'] = auc
#     dic_nested_roc[model_mass] = dic_roc            


    #-- implement systematics --#   
    os.system("mkdir -p Outputs/"+REGION)
    os.system("mkdir -p Outputs/"+REGION+"/sys")
    sys_list = ["puweight", "l1prefiring", "effSF", "topSF"]
    # sys_list = ["puweight"]

    r = [nbin, 0, 1, "DNN score"]

    if REGION == 'SR':
        threshold = 0.3
    else:
        threshold = 1    
    
    # draw plot with syst
    dic_sys_sqr = draw_with_systematics(dic_predict_sort, sys_list, "score"+model_mass, r, "region_dummy")

    hist=HistStack(dic_predict_sort, "score"+model_mass, 'weight', nbin, 0, 1, "DNN score", dic_sys=dic_sys_sqr) 
    
    c1=hist.draw_hist(model_mass, ERA, file_name="Outputs/"+REGION+"/"+model_mass, blind_from=threshold)     
    hist.get_yeilds()
    hist.draw_pie(file_name="Outputs/"+REGION+"/"+model_mass)     
     
    return None

# Load model and evaluate

In [9]:
import time

nbin=20
# VERSION = '0903'
# VERSION = setting["VERSION"]
print(time.time())

one_mass = 'Zp-2500_CH-345'

if 'mm' in INPUTDIR:
    CHANNEL = 'mm'
if 'ee' in INPUTDIR:
    CHANNEL = 'ee'
if 'em' in INPUTDIR:
    CHANNEL = 'em'
    
OUTDIR = 'Outputs/'+CHANNEL
file_postfix = 'PNN_'+VERSION
# file_postfix = one_mass+VERSION

# model_masses = ['Zp-1700_CH-345', 'Zp-2500_CH-345', 'Zp-2900_CH-345', 'Zp-3300_CH-345', 'Zp-4100_CH-345']
model_masses = [m for m in dic_df_train if 'Zp' in m] # if want to run all
# model_masses = ['Zp-2500_CH-845']

# %matplotlib inline
rt.gROOT.SetBatch(rt.kTRUE)

dic_nested_roc = {}

ERA = 'ee17'

model = keras.models.load_model(OUTDIR+'/model_'+file_postfix)
#     model = keras.models.load_model(OUTDIR+'/model_binary_'+file_postfix)

Transformer = pickle.load(open(OUTDIR+'/scaler_'+file_postfix+'.pkl','rb'))

nested_dic_predict, nested_dic_predict_met_u, nested_dic_predict_met_d = [{} for _ in range(3)]

# masses loop
for model_mass in model_masses:
    masses = model_mass.split('_')
    zp_m = masses[0].split('-')
    zp_m = zp_m[1]
    ch_m = masses[1].split('-')
    ch_m = ch_m[1]
    
#     file_postfix = 'test'

    dic_df_scale, dic_df_scale_met_u, dic_df_scale_met_d = [{} for _ in range(3)]        

    for key2 in dic_df_train:
        dic_df_train[key2]['zp_m'] = np.full(dic_df_train[key2].shape[0], int(zp_m))
        dic_df_train[key2]['ch_m'] = np.full(dic_df_train[key2].shape[0], int(ch_m))
        dic_df_met_u[key2]['zp_m'] = np.full(dic_df_met_u[key2].shape[0], int(zp_m))
        dic_df_met_u[key2]['ch_m'] = np.full(dic_df_met_u[key2].shape[0], int(ch_m))
        dic_df_met_d[key2]['zp_m'] = np.full(dic_df_met_d[key2].shape[0], int(zp_m))
        dic_df_met_d[key2]['ch_m'] = np.full(dic_df_met_d[key2].shape[0], int(ch_m))
        dic_df_scale[key2] = Transformer.transform(dic_df_train[key2]) 
        dic_df_scale_met_u[key2] = Transformer.transform(dic_df_met_u[key2]) 
        dic_df_scale_met_d[key2] = Transformer.transform(dic_df_met_d[key2]) 

    # predict
    dic_predict, dic_predict_met_u, dic_predict_met_d = [{} for _ in range(3)]

    for key3 in dic_df_scale:
    #             if "Zp-" in key3 and not key in key3: # use same scaling used in model training for all masses
    #                 continue
        dic_predict[key3] = model.predict(dic_df_scale[key3], batch_size=dic_df_scale[key3].shape[0])
        dic_predict_met_u[key3] = model.predict(dic_df_scale_met_u[key3], batch_size=dic_df_scale_met_u[key3].shape[0])
        dic_predict_met_d[key3] = model.predict(dic_df_scale_met_d[key3], batch_size=dic_df_scale_met_d[key3].shape[0])
    
    nested_dic_predict[model_mass] = dic_predict
    nested_dic_predict_met_u[model_mass] = dic_predict_met_u
    nested_dic_predict_met_d[model_mass] = dic_predict_met_d
    

# array in y-axis to dataframe
dic_predict_sort, dic_predict_met_u_sort, dic_predict_met_d_sort = [{} for _ in range(3)]

for key4 in dic_df:
    new_df = dic_df[key4].copy(deep=True).reset_index(drop="True")
    new_df_met_u = pd.DataFrame()
    new_df_met_d = pd.DataFrame()
    new_df_met_u["weightmetUp"] = dic_df_met_u_unsort[key4]["weight"].copy(deep=True).reset_index(drop="True")
    new_df_met_d["weightmetDown"] = dic_df_met_d_unsort[key4]["weight"].copy(deep=True).reset_index(drop="True")    
    dic_predict_sort[key4] = new_df
    dic_predict_met_u_sort[key4] = new_df_met_u
    dic_predict_met_d_sort[key4] = new_df_met_d
    
for model_mass in model_masses:
    for key5 in dic_df:
        dic_predict_sort[key5]["score"+model_mass] = nested_dic_predict[model_mass][key5][:,0]
        dic_predict_met_u_sort[key5]["score"+model_mass] = nested_dic_predict_met_u[model_mass][key5][:,0]
        dic_predict_met_d_sort[key5]["score"+model_mass] = nested_dic_predict_met_d[model_mass][key5][:,0]
    
        
with Pool() as p:
    p.map(wrapper_mp, model_masses)  


with open(OUTDIR+'/nested_dic_roc_'+file_postfix+'.pkl', 'wb') as f:
    pickle.dump(dic_nested_roc, f)            
        


1680288896.0905793


/tmp/ipykernel_161854/1643505186.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_train[key2]['zp_m'] = np.full(dic_df_train[key2].shape[0], int(zp_m))
/tmp/ipykernel_161854/1643505186.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_train[key2]['ch_m'] = np.full(dic_df_train[key2].shape[0], int(ch_m))
/tmp/ipykernel_161854/1643505186.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instea

/tmp/ipykernel_161854/1643505186.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_train[key2]['zp_m'] = np.full(dic_df_train[key2].shape[0], int(zp_m))
/tmp/ipykernel_161854/1643505186.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_train[key2]['ch_m'] = np.full(dic_df_train[key2].shape[0], int(ch_m))
/tmp/ipykernel_161854/1643505186.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instea

/tmp/ipykernel_161854/1643505186.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_train[key2]['zp_m'] = np.full(dic_df_train[key2].shape[0], int(zp_m))
/tmp/ipykernel_161854/1643505186.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_train[key2]['ch_m'] = np.full(dic_df_train[key2].shape[0], int(ch_m))
/tmp/ipykernel_161854/1643505186.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instea

/tmp/ipykernel_161854/1643505186.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_train[key2]['zp_m'] = np.full(dic_df_train[key2].shape[0], int(zp_m))
/tmp/ipykernel_161854/1643505186.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_train[key2]['ch_m'] = np.full(dic_df_train[key2].shape[0], int(ch_m))
/tmp/ipykernel_161854/1643505186.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instea

...

         3.85388907,   4.08762346,   3.62355891,   3.38020573])}
{'puweightUp': array([17.05114474,  0.78711296,  7.39372226, 10.68954867,  4.60909829,
       -2.1216895 , -0.90221131, -4.18978015,  2.89155433,  2.39634541,
        0.58308921,  1.49517846,  1.00486782,  0.41220518,  1.15707261,
        0.08689282,  0.96502796, -0.87138397,  0.60808845, -0.90266235]), 'puweightDown': array([-16.82401844,  -2.56235751,  -5.73104084, -10.39553051,
        -3.19457299,   1.03212207,   0.73702086,   1.93778981,
        -0.46225102,  -2.83415945,  -1.05834377,  -1.79744157,
        -0.70258323,  -0.23906977,  -0.80945836,  -0.59442894,
        -0.62982998,   0.92720861,  -0.58435755,   0.50728001]), 'l1prefiringUp': array([0.2477667 , 0.15861319, 0.09479438, 0.06485671, 0.04644402,
       0.03855531, 0.03683135, 0.02741864, 0.02320979, 0.02736091,
       0.01845239, 0.01563608, 0.01391486, 0.01100997, 0.01274761,
       0.01020807, 0.01514384, 0.01586981, 0.011606  , 0.01231187]), 'l1prefiri

        5.76945443,  6.43060467,  7.24648608,  5.90719667,  4.99909699])}
{'puweightUp': array([37.62116301,  3.54642832,  6.05119356,  1.16815528, -0.53958377,
       -2.40174799, -0.07257245, -0.23277847, -0.79953645,  0.42949203,
        0.89505713, -0.49055714, -0.46084927, -0.04139405,  0.43713477,
       -1.40446083, -0.69359746,  0.68928009,  0.09481978, -0.65242416]), 'puweightDown': array([-3.48771767e+01, -5.85029664e+00, -5.82965201e+00, -1.13262229e+00,
        2.74179180e-01,  2.25690286e+00, -3.26765271e-02, -1.61757700e-02,
        7.88418092e-01, -5.42068448e-01, -9.63614206e-01,  9.56477678e-01,
        7.37885334e-01, -3.90937653e-02, -3.50177646e-01,  1.15538994e+00,
        7.96294244e-01, -6.25686643e-01, -4.58152360e-01,  4.73823101e-01]), 'l1prefiringUp': array([0.53589683, 0.12982086, 0.05295914, 0.02982785, 0.02116036,
       0.01611979, 0.01294294, 0.01025674, 0.00933775, 0.0085709 ,
       0.00874213, 0.00738588, 0.00643722, 0.00645464, 0.00706452,
       0.0

         3.95457172,   5.04904682,   4.30563152,   3.40077269])}
{'puweightUp': array([ 4.74018982e+01,  3.77454290e-01,  2.68773429e+00, -5.05810474e+00,
       -2.09609779e-02, -1.69682564e-01,  2.12165514e-01,  7.10566259e-01,
        4.16134232e-01, -1.85782004e+00, -7.16449208e-01,  6.02795302e-02,
       -3.43121488e-01, -2.59817416e-01, -1.20218480e-01,  2.52220308e-01,
       -4.38587983e-03,  8.55384928e-01, -3.74736507e-01, -9.05318341e-01]), 'puweightDown': array([-47.03372943,  -0.50617161,  -0.98531494,   2.96755747,
         0.24451977,   0.42601701,  -0.28227009,  -0.53290663,
        -0.18005795,   1.72374807,   0.78582683,  -0.13929052,
         0.28658499,   0.19886646,   0.19097885,  -0.21306881,
        -0.0602115 ,  -1.1222541 ,   0.33507042,   0.61808312]), 'l1prefiringUp': array([0.70604212, 0.0720347 , 0.02691574, 0.01587821, 0.01180061,
       0.0086297 , 0.0049169 , 0.00435923, 0.00462682, 0.00365031,
       0.0041565 , 0.00423351, 0.00480389, 0.00390543, 0.00

         2.35164346,   2.96585946,   2.84130637,   2.65436511])}
array('f', [16.376644134521484, 17.94869041442871, 26.768770217895508, 425.4893798828125, 964.8348999023438, 13.330436706542969, 9084.0693359375, 414.8154602050781, 19.586641311645508])
test
array('f', [16.376644134521484, 17.94869041442871, 26.768770217895508, 425.4893798828125, 964.8348999023438, 13.330436706542969, 9084.0693359375, 414.8154602050781, 19.586641311645508])
test
array('f', [16.376644134521484, 17.94869041442871, 26.768770217895508, 425.4893798828125, 964.8348999023438, 13.330436706542969, 9084.0693359375, 414.8154602050781, 19.586641311645508])
test
array('f', [16.376644134521484, 17.94869041442871, 26.768770217895508, 425.4893798828125, 964.8348999023438, 13.330436706542969, 9084.0693359375, 414.8154602050781, 19.586641311645508])
test
array('f', [16.376644134521484, 17.94869041442871, 26.768770217895508, 425.4893798828125, 964.8348999023438, 13.330436706542969, 9084.0693359375, 414.8154602050781, 19.586

2023-03-31 20:54:56.494967: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /cvmfs/sft.cern.ch/lcg/releases/MCGenerators/thepeg/2.2.3-6cb32/x86_64-centos7-gcc11-opt/lib/ThePEG:/cvmfs/sft.cern.ch/lcg/releases/MCGenerators/herwig++/7.2.3-90caf/x86_64-centos7-gcc11-opt/lib/Herwig:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/jaxlib/mlir/_mlir_libs:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/torch/lib:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/onnxruntime/capi/:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/tensorflow:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/tensorflow/contrib